In [1]:
import os
import re

# This script is used to set the working directory to the root of the project.
cwd = os.getcwd()
project_name = 'braacket-scraper'
project_root = re.split(rf'(?!.*({project_name}))/(.+)$', cwd)[0]
os.chdir(project_root)
project_root

'/home/agiera/ssbm/braacket-scraper'

In [2]:
import polars as pl
import rating_utils
from IPython.display import display, HTML


In [3]:
year_before_season = '2023-11-03'
season_start = '2025-04-20'
season_end = '2025-10-18'

In [4]:
candidates = pl.DataFrame([
    ["/league/nemelee/player/11CA20C1-5FDF-4A64-BC6C-0B5B91497FE7?", "MATE | 22K"],
    ["/league/nemelee/player/5DBCC342-92B6-40C6-AFBF-66933EF41379?", "Ant"],
    ["/league/nemelee/player/E85EE0F4-C335-4A1C-A569-BE69C1675BAA?", "$G|MP | Bank"],
    ["/league/nemelee/player/583438BD-F9A1-413A-8F77-DEF3820143F0?", "N0va | hc | Coolslice"],
    ["/league/nemelee/player/BAB54F38-444F-4ADE-8D79-EC7D232D90E9?", "OUG | Electroman"],
    ["/league/nemelee/player/5CD3DDFC-6380-4395-8DF7-590D498437E2?", "Future Shock"],
    ["/league/nemelee/player/0267F6B4-223B-4D0D-B0AB-3B7084B37AB7?", "Greenstach"],
    ["/league/nemelee/player/EAE5E0CC-99FF-49EB-8424-142348454761?", "Comcast | The CoveCare Scare"],
    ["/league/nemelee/player/3576ECE4-EACF-4E83-995C-F40FB232EADD?", 'Hysteric'],
    ["/league/nemelee/player/0CDB11B1-BCC3-4027-90F2-D74F1A9A5668?", "hc | kraft"],
    ["/league/nemelee/player/0D8C1F36-D79B-4FEF-91DE-5963AC99D13D?", "MEAT"],
    ["/league/nemelee/player/E12030B6-0105-48C8-8D9F-C89581A6F3B9?", 'Motobug'],
    ["/league/nemelee/player/8FF9A4C4-C07C-4DB0-9BAC-9549C3DEB401?", 'oz'],
    ["/league/nemelee/player/1D8B9E33-86AE-45FB-89E9-0CB158C24FD3?", "regEx"],
    ["/league/nemelee/player/2407F41F-64A0-43AF-85F1-96F6985E57EE?", "hc | saucymain"],
    ["/league/nemelee/player/DF3CE0B4-0BA0-493F-99F3-879CD12469FD?", 'Sweat'],
    ["/league/nemelee/player/62EEC119-0B50-4FC8-BE56-F566A076EB0A?", "Gambit"],
], orient="row", schema=["url", "tag"])
candidate_urls = candidates.select('url').to_series().to_list()


# Load Data

In [5]:
matches = pl.read_csv("data/matches.csv")
season_matches = (
    matches
    .filter(pl.col('tournament_date') >= season_start)
    .filter(pl.col('tournament_date') <= season_end)
)

## Using just season data

In [6]:
season_matches_with_ratings = rating_utils.calculate_ratings(matches, season_start, season_end)
season_player_ratings = rating_utils.get_final_ratings(season_matches_with_ratings)
pl.Config(tbl_rows=20)
season_player_ratings.filter(pl.col('url').str.contains_any(candidate_urls))

/home/agiera/ssbm/braacket-scraper/rating_utils.py:147: MapWithoutReturnDtypeWarning: Calling `map_elements` without specifying `return_dtype` can lead to unpredictable results. Specify `return_dtype` to silence this warning.
  .with_columns(


url,starting_rating,rating,rating_95ci_lower,rating_95ci_upper,sigma,pi,tau,exposure,tournament_date,encounter_id,tag
str,f64,f64,f64,f64,f64,f64,f64,f64,str,i64,str
"""/league/nemelee/player/BAB54F3…",35.990487,36.230861,32.545465,39.916258,1.842698,0.294504,10.670144,30.702767,"""2025-10-18""",87,"""OUG | Electroman"""
"""/league/nemelee/player/1D8B9E3…",39.668469,35.475246,28.592827,42.357666,3.44121,0.084446,2.995733,25.151617,"""2025-10-14""",70,"""regEx"""
"""/league/nemelee/player/583438B…",35.002718,35.282463,32.967783,37.597143,1.15734,0.746583,26.341285,31.810443,"""2025-10-18""",51,"""N0va | hc | Coolslice"""
"""/league/nemelee/player/2407F41…",34.158758,34.438089,29.575058,39.301119,2.431515,0.16914,5.824854,27.143543,"""2025-10-18""",83,"""hc | saucymain"""
"""/league/nemelee/player/EAE5E0C…",34.069209,34.087749,32.250819,35.92468,0.918465,1.185426,40.408507,31.332354,"""2025-10-18""",89,"""Comcast | The CoveCare Scare"""
"""/league/nemelee/player/0D8C1F3…",33.551028,33.731655,31.090296,36.373014,1.320679,0.573331,19.33939,29.769617,"""2025-10-05""",126,"""MEAT"""
"""/league/nemelee/player/5CD3DDF…",33.722688,33.564796,31.796368,35.333223,0.884214,1.279044,42.930845,30.912154,"""2025-10-18""",86,"""Future Shock"""
"""/league/nemelee/player/5DBCC34…",33.63362,33.503327,31.506919,35.499735,0.998204,1.003602,33.62399,30.508715,"""2025-10-07""",70,"""Ant"""
"""/league/nemelee/player/3576ECE…",32.769562,32.553059,29.957226,35.148892,1.297917,0.593617,19.324056,28.65931,"""2025-10-07""",71,"""Hysteric"""


# Using data outside season for non candidates who didn't enter enough

In [7]:
matches_with_ratings = rating_utils.calculate_ratings(
    matches.filter(pl.col('tournament_date') > year_before_season),
    season_start, season_end, sigma_fix=2.0,
)

/home/agiera/ssbm/braacket-scraper/rating_utils.py:147: MapWithoutReturnDtypeWarning: Calling `map_elements` without specifying `return_dtype` can lead to unpredictable results. Specify `return_dtype` to silence this warning.
  .with_columns(
/home/agiera/ssbm/braacket-scraper/rating_utils.py:147: MapWithoutReturnDtypeWarning: Calling `map_elements` without specifying `return_dtype` can lead to unpredictable results. Specify `return_dtype` to silence this warning.
  .with_columns(
/home/agiera/ssbm/braacket-scraper/rating_utils.py:147: MapWithoutReturnDtypeWarning: Calling `map_elements` without specifying `return_dtype` can lead to unpredictable results. Specify `return_dtype` to silence this warning.
  .with_columns(


In [8]:
player_ratings = rating_utils.get_final_ratings(matches_with_ratings)

In [9]:
pl.Config(tbl_rows=20)
candidate_placings = (
    player_ratings
    .filter(pl.col('url').str.contains_any(candidate_urls))
    .select(pl.exclude('starting_rating', 'url', 'pi', 'tau', 'exposure', 'encounter_id'))
)
candidate_placings

rating,rating_95ci_lower,rating_95ci_upper,sigma,tournament_date,tag
f64,f64,f64,f64,str,str
39.002443,35.444615,42.560271,1.778914,"""2025-10-18""","""OUG | Electroman"""
38.734664,36.976077,40.49325,0.879293,"""2025-10-18""","""$G|MP | Bank"""
38.045804,36.327417,39.764191,0.859193,"""2025-10-14""","""regEx"""
36.715869,34.369503,39.062235,1.173183,"""2025-10-18""","""N0va | hc | Coolslice"""
36.433685,34.732805,38.134565,0.85044,"""2025-10-18""","""Greenstach"""
35.445605,33.68859,37.202619,0.878507,"""2025-10-18""","""hc | saucymain"""
35.246728,32.550986,37.94247,1.347871,"""2025-10-05""","""MEAT"""
34.839386,33.017637,36.661136,0.910875,"""2025-10-18""","""Comcast | The CoveCare Scare"""
34.366552,32.350573,36.38253,1.007989,"""2025-10-07""","""Ant"""


In [10]:
matches_with_difference = (
    matches_with_ratings
    .join(player_ratings.select('url', pl.col('rating').alias('winner_final_rating')), left_on='winner_url', right_on='url')
    .join(player_ratings.select('url', pl.col('rating').alias('loser_final_rating')), left_on='loser_url', right_on='url')
    .with_columns((pl.col('winner_final_rating') - pl.col('loser_final_rating')).alias('final_rating_difference'))
    .select('winner_url', 'winner', 'winner_final_rating', 'loser_url', 'loser', 'loser_final_rating', 'final_rating_difference')
)

# Match Analysis


If some player's rating is higher $\beta$ than another player's, the player may have about a 76% (specifically $\Phi(\frac {1}{\sqrt{2}}))$ chance to beat the other player. The default value of $\beta$ is $\frac{ 25 }{ 6 }$.

## Wins in order of rating difference

In [11]:
def best_wins(matches: pl.DataFrame, player_url: str) -> pl.DataFrame:
    return (
        matches
        .filter(pl.col('winner_url').str.contains(player_url))
        .sort('final_rating_difference')
        .select(pl.exclude(r'^.*url$'))
        .select(pl.exclude(r'^winner.*$'))
        .head(100)
    )

## Losses in order of rating difference

In [12]:
def worst_losses(matches: pl.DataFrame, player_url: str) -> pl.DataFrame:
    return (
        matches
        .filter(pl.col('loser_url').str.contains(player_url))
        .select(pl.exclude(r'^.*url$'))
        .select(pl.exclude(r'^loser.*$'))
        .sort('final_rating_difference')
    )

In [13]:
def display_side_by_side(*args, titles=('',)):
    html_str = ''
    if len(titles) > 0:
        html_str += '<div style="display:flex">'
    for df, title in zip(args, titles + ('',) * (len(args) - len(titles))):
        html_str += '<div style="margin-right:20px">'
        if title:
            html_str += f'<h2>{title}</h2>'
        html_str += df.to_html()
        html_str += '</div>'
    html_str += '</div>'
    display(HTML(html_str))

In [14]:
def display_player_comparison(matches: pl.DataFrame, player1: str, player2: str) -> pl.DataFrame:
    display_side_by_side(
        best_wins(matches, player1).to_pandas(),
        best_wins(matches, player2).to_pandas(),
    )
    display_side_by_side(
        worst_losses(matches, player1).to_pandas(),
        worst_losses(matches, player2).to_pandas(),
    )


In [15]:
pl.Config(tbl_rows=100)
for i in range(0, len(candidate_urls) - 1):
    first_candidate = candidate_placings.slice(i, 1).select('tag').item()
    second_candidate = candidate_placings.slice(i+1, 1).select('tag').item()
    print(f"Comparing {first_candidate} and {second_candidate}")
    display_player_comparison(matches_with_difference, candidate_urls[i], candidate_urls[i+1])

Comparing OUG | Electroman and $G|MP | Bank


,loser,loser_final_rating,final_rating_difference
0,Team Porp | Dr. Lame,36.854337,-5.014399
1,Greenstach,36.433685,-4.593748
2,Greenstach,36.433685,-4.593748
3,Ant,34.366552,-2.526615
4,UConn | kuro,34.323890,-2.483953
5,Motobug,31.980652,-0.140714
6,Gambit,31.725166,0.114771
7,Gambit,31.725166,0.114771
8,Gambit,31.725166,0.114771
9,Gambit,31.725166,0.114771


,winner,winner_final_rating,final_rating_difference
0,N0va | WAS | N0va,20.337846,-11.502092
1,Gambit,31.725166,-0.114771
2,Gambit,31.725166,-0.114771
3,Gambit,31.725166,-0.114771
4,Motobug,31.980652,0.140714
5,KevinM,33.339548,1.499611
6,KevinM,33.339548,1.499611
7,KevinM,33.339548,1.499611
8,Boilerguy,34.236224,2.396287
9,The Arbiter,34.323890,2.483953


Comparing $G|MP | Bank and regEx


,loser,loser_final_rating,final_rating_difference
0,Younger,39.505808,-5.139256
1,OUG | Electroman,39.002443,-4.635892
2,OUG | Electroman,39.002443,-4.635892
3,regEx,38.045804,-3.679253
4,Karils | Conflict,36.803742,-2.437190
5,N0va | hc | Coolslice,36.715869,-2.349317
6,N0va | hc | Coolslice,36.715869,-2.349317
7,hc | saucymain,35.445605,-1.079053
8,hc | saucymain,35.445605,-1.079053
9,Go Sox | Future Shock,34.159223,0.207329


...

,loser,loser_final_rating,final_rating_difference
0,Younger,39.505808,-4.060204
1,OUG | Electroman,39.002443,-3.556839
2,OUG | Electroman,39.002443,-3.556839
3,$G|MP | Bank,38.734664,-3.289059
4,Abel,38.007105,-2.561500
5,Team Porp | Dr. Lame,36.854337,-1.408732
6,Karils | Conflict,36.803742,-1.358137
7,Karils | Conflict,36.803742,-1.358137
8,Karils | Conflict,36.803742,-1.358137
9,Karils | Conflict,36.803742,-1.358137


,winner,winner_final_rating,final_rating_difference
0,Kumatora,28.360636,-7.084968
1,oz,31.003236,-4.442368
2,oz,31.003236,-4.442368
3,Gambit,31.725166,-3.720439
4,Gambit,31.725166,-3.720439
5,Gambit,31.725166,-3.720439
6,Artelind,31.843204,-3.602401
7,HABS | Top Player Yasu,31.976639,-3.468966
8,heeg | meep,32.419748,-3.025857
9,hc | kraft,33.368574,-2.077030


Comparing Gambit and oz


,loser,loser_final_rating,final_rating_difference
0,Greenstach,36.433685,-2.935095
1,Greenstach,36.433685,-2.935095
2,UConn | isaac,34.323890,-0.825300
3,Future Shock,34.159223,-0.660633
4,Padre,33.916909,-0.418319
5,Gambit,31.725166,1.773424
6,Gambit,31.725166,1.773424
7,zeldEx,30.491597,3.006993
8,Twisty,29.696733,3.801857
9,Raf,29.545518,3.953072


,winner,winner_final_rating,final_rating_difference
0,Nico,26.786057,-6.712532
1,oz,31.003236,-2.495354
2,Padre,33.916909,0.418319
3,kuro,34.323890,0.825300
4,Comcast | The CoveCare Scare,34.839386,1.340797
5,hc | saucymain,35.445605,1.947015
6,hc | saucymain,35.445605,1.947015
7,Greenstach,36.433685,2.935095
8,Greenstach,36.433685,2.935095
9,Karils | Conflict,36.803742,3.305152
